# TreeStack-CNN on Kaggle or Colab

This notebook runs the leakage-free TreeStack experiment on NVIDIA GPUs. On Kaggle's **T4 x2** accelerator, two CNNs train at the same time, one per GPU. The third CNN starts as soon as either GPU becomes free. Each training process receives one data-loader worker, which leaves enough headroom for Kaggle's four CPU cores.

The default run uses one Fashion-MNIST seed. It is a convergence test, not the final paper result. After the individual CNNs reach credible accuracy, change `SEEDS` to `[17, 42, 73]` and run both datasets.

## 1. Choose the run

In Kaggle, open **Settings → Accelerator → GPU T4 x2** before running the notebook. Colab usually exposes one GPU; the same runner then trains the CNNs sequentially on that device.

In [ ]:
DATASET = "fashion_mnist"       # "fashion_mnist" or "cifar10"
SEEDS = [42]                      # Publication run: [17, 42, 73]
EPOCHS = 25                       # Fashion: 25; CIFAR-10: 120
BATCH_SIZE = 512                  # Per training process/GPU
MAX_GPUS = 2
NUM_WORKERS = 1                   # Two trainers × one worker fits a 4-core CPU
FORCE_RETRAIN = False

REPOSITORY = "https://github.com/Prathmesh333/WeakEnsembleStrongDecisionTree.git"
print({"dataset": DATASET, "seeds": SEEDS, "epochs": EPOCHS})

## 2. Fetch and activate the project

Kaggle must have Internet enabled for the clone step. If Internet is disabled, add the GitHub repository as a Kaggle Dataset and set `REPO_DIR` below to that mounted folder. This cell adds the source directory to the current kernel and to each training subprocess. A kernel restart is not required. Generated datasets and results stay in the notebook's writable working directory.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working" if ON_KAGGLE else "/content")
REPO_DIR = WORK_ROOT / "WeakEnsembleStrongDecisionTree"

if not (REPO_DIR / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
SRC_DIR = (REPO_DIR / "src").resolve()
runner_file = SRC_DIR / "treestack_cnn" / "cuda_runner.py"
assert runner_file.exists(), (
    f"The checked-out commit {commit} does not contain {runner_file.name}. "
    "Delete the repository directory. Then rerun this cell."
)

# Activate the src-layout package in this running notebook kernel.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
pythonpath = [str(SRC_DIR)]
pythonpath.extend(
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep)
    if entry and entry != str(SRC_DIR)
)
os.environ["PYTHONPATH"] = os.pathsep.join(pythonpath)
RUN_ENV = os.environ.copy()

# Remove an older cached checkout before importing the current source.
for module_name in list(sys.modules):
    if module_name == "treestack_cnn" or module_name.startswith("treestack_cnn."):
        del sys.modules[module_name]
importlib.invalidate_caches()
runner_module = importlib.import_module("treestack_cnn.cuda_runner")
loaded_runner = Path(runner_module.__file__).resolve()
assert loaded_runner == runner_file.resolve(), (
    f"The kernel loaded {loaded_runner}, not {runner_file}. Restart the session and rerun this cell."
)
subprocess.run(
    [sys.executable, "-c", "import treestack_cnn.cuda_runner as m; print('Subprocess import:', m.__file__)"],
    cwd=REPO_DIR, env=RUN_ENV, check=True,
)
print(f"Kernel import: {loaded_runner}")
print(f"Project directory: {REPO_DIR} (commit {commit})")

## 3. Inspect the diversity sources

Diversity is allowed in any defensible form: layer structure, width, residual connections, regularization, optimizer, learning rate or initialization. This run uses several of those at once. The data split and evaluation protocol stay identical so the comparison remains fair.

In [ ]:
import pandas as pd
from IPython.display import display

from treestack_cnn.cuda_runner import MODEL_DIVERSITY_PROFILES
from treestack_cnn.models import build_models
from treestack_cnn.utils import count_parameters

in_channels = 1 if DATASET == "fashion_mnist" else 3
models = build_models(in_channels=in_channels, num_classes=10)
diversity_rows = []
for model_name, model in models.items():
    profile = MODEL_DIVERSITY_PROFILES[model_name]
    diversity_rows.append({
        "model": model_name,
        "parameters": count_parameters(model),
        "optimizer": profile["optimizer"],
        "lr_multiplier": profile["learning_rate_multiplier"],
        "weight_decay": profile["weight_decay"],
        "label_smoothing": profile["label_smoothing"],
        "diversity_source": profile["diversity_source"],
    })
diversity_table = pd.DataFrame(diversity_rows)
display(diversity_table)
del models

## 4. Audit the leakage boundary before training

The following cell builds the exact seeded split and proves that the base, meta and test indices do not overlap. CNN early stopping uses only the validation portion inside the base partition.

In [ ]:
from treestack_cnn.config import DatasetConfig
from treestack_cnn.data import build_dataset

DATA_ROOT = WORK_ROOT / "treestack-data"
audit_config = DatasetConfig(name=DATASET, root=str(DATA_ROOT), num_workers=0, download=True)
audit_bundle = build_dataset(audit_config, SEEDS[0])
split_sets = {
    "base": set(audit_bundle.splits.base.tolist()),
    "meta": set(audit_bundle.splits.meta.tolist()),
    "test": set(audit_bundle.splits.test.tolist()),
}
assert not (split_sets["base"] & split_sets["meta"])
assert not (split_sets["base"] & split_sets["test"])
assert not (split_sets["meta"] & split_sets["test"])
split_table = {
    "complete": sum(map(len, split_sets.values())),
    "base_total": len(audit_bundle.splits.base),
    "base_train": len(audit_bundle.splits.base_train),
    "base_validation": len(audit_bundle.splits.base_validation),
    "meta_train": len(audit_bundle.splits.meta),
    "final_test": len(audit_bundle.splits.test),
}
print(split_table)
print("Leakage audit: PASS")
del audit_bundle, split_sets

## 5. Verify the accelerator

The runner uses separate spawned processes rather than notebook-defined multiprocessing functions. This avoids CUDA's unsafe `fork` behavior and the pickling failures that often occur when workers are defined inside a notebook cell.

In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=False)
GPU_COUNT = torch.cuda.device_count()
print(f"PyTorch {torch.__version__}; CUDA devices: {GPU_COUNT}")
for index in range(GPU_COUNT):
    properties = torch.cuda.get_device_properties(index)
    print(f"  cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")
assert GPU_COUNT > 0, "No CUDA GPU found. Enable a GPU accelerator and restart the session."
if GPU_COUNT == 1:
    print("One GPU detected: models will train one after another.")
else:
    print("Multiple GPUs detected: independent CNNs will train concurrently.")

## 6. Train the CNNs and fit the ensemble

The command performs the complete 60/20/20 experiment: CNN training, out-of-sample probability generation, voting baselines, logistic stacking, decision-tree variants, ablations and report generation. Checkpoints are reused unless `FORCE_RETRAIN` is enabled.

In [ ]:
OUTPUT_DIR = WORK_ROOT / "treestack-results"
DATA_ROOT = WORK_ROOT / "treestack-data"
command = [
    sys.executable, "-m", "treestack_cnn.cuda_runner",
    "--dataset", DATASET,
    "--seeds", *map(str, SEEDS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--max-gpus", str(min(MAX_GPUS, GPU_COUNT)),
    "--num-workers", str(NUM_WORKERS),
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(OUTPUT_DIR),
]
if FORCE_RETRAIN:
    command.append("--force")

print("Running:", " ".join(command))
subprocess.run(command, cwd=REPO_DIR, env=RUN_ENV, check=True)

## 7. Inspect the result table

Do not decide from accuracy alone. DT-Soft should be compared with the strongest CNN, soft voting and logistic stacking. The standard deviation becomes meaningful only after all three seeds are complete.

In [ ]:
aggregate = pd.read_csv(OUTPUT_DIR / "aggregate_results.csv")
main_results = aggregate[aggregate["category"] == "main"].copy()
main_results["accuracy_percent"] = 100 * main_results["accuracy_mean"]
main_results["macro_f1_percent"] = 100 * main_results["macro_f1_mean"]
display(
    main_results[[
        "method", "seeds", "accuracy_percent", "accuracy_std",
        "macro_f1_percent", "inference_ms_mean"
    ]].sort_values("accuracy_percent", ascending=False)
)
display(pd.read_csv(OUTPUT_DIR / "paper_accuracy_table.csv"))

## 8. Inspect one run in detail

The aggregate table is useful for comparison across seeds. The per-run report below records the GPU schedule, validation scores, deliberately different training profiles and fitted decision-tree settings.

In [ ]:
import json
from IPython.display import Image, display

run_reports = sorted(OUTPUT_DIR.glob(f"{DATASET}/seed_*/**/report.json"))
assert run_reports, f"No reports found below {OUTPUT_DIR}. The training cell may have failed."
latest_report = run_reports[-1]
run_dir = latest_report.parent
figure_dir = run_dir / "figures"
report = json.loads(latest_report.read_text(encoding="utf-8"))
print("Report:", latest_report)
print("GPU assignments:", report["model_gpu_assignments"])
print("Validation accuracies:", report["validation_accuracies"])
print("DT-Soft parameters:", report["combiner_best_parameters"]["dt_soft"])
display(pd.DataFrame(report["training_profiles"]).T.rename_axis("model"))

## 9. Check convergence and overfitting

A diverse model can still overfit. These curves show whether each CNN converged and how far its training accuracy moved away from validation accuracy. Model choices should be made from these validation histories, never from final-test performance.

In [ ]:
import matplotlib.pyplot as plt

histories = {}
convergence_rows = []
for model_name in report["training_profiles"]:
    history_path = run_dir / "training" / f"{model_name}.csv"
    history = pd.read_csv(history_path)
    histories[model_name] = history
    best_index = history["validation_accuracy"].idxmax()
    best = history.loc[best_index]
    convergence_rows.append({
        "model": model_name,
        "epochs_run": len(history),
        "best_epoch": int(best["epoch"]),
        "best_validation_accuracy": best["validation_accuracy"],
        "train_accuracy_at_best_epoch": best["train_accuracy"],
        "train_validation_gap": best["train_accuracy"] - best["validation_accuracy"],
    })

display(pd.DataFrame(convergence_rows).sort_values("best_validation_accuracy", ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for model_name, history in histories.items():
    axes[0].plot(history["epoch"], history["train_accuracy"], linestyle="--", alpha=0.65, label=f"{model_name} train")
    axes[0].plot(history["epoch"], history["validation_accuracy"], label=f"{model_name} validation")
    axes[1].plot(history["epoch"], history["validation_loss"], label=model_name)
axes[0].set(title="Training and validation accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].set(title="Validation loss", xlabel="Epoch", ylabel="Cross-entropy")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 10. Decision-tree depth ablation

The meta learner should earn its complexity. This ablation compares shallow and unrestricted trees using the same held-out meta-training set and final test set.

In [ ]:
run_results = pd.read_csv(run_dir / "results.csv")
depth_results = run_results[run_results["category"] == "depth_ablation"].copy()
depth_results["accuracy_percent"] = 100 * depth_results["accuracy"]
display(depth_results[[
    "method", "accuracy_percent", "macro_f1",
    "tree_depth", "tree_leaves", "combiner_parameters"
]].sort_values("accuracy_percent", ascending=False))

## 11. Measure useful diversity

Architecture and hyperparameter differences are only proposed sources of diversity. The quantities below test whether they actually produced different predictions and complementary errors on the untouched test set. A lower double-fault rate and a higher oracle accuracy indicate more useful complementarity.

In [ ]:
pairwise = report["pairwise_diversity_analysis"]
pairwise_table = pd.DataFrame.from_dict(pairwise["pairwise"], orient="index")
pairwise_table.index.name = "model_pair"
display(pairwise_table)
print(f"Oracle accuracy (at least one CNN correct): {pairwise['oracle_accuracy_any_cnn_correct']:.4f}")

correction = report["disagreement_analysis"]
display(pd.DataFrame([correction]).T.rename(columns={0: "samples"}))
net_corrections = correction["tree_corrects_majority"] - correction["tree_harms_correct_majority"]
print(f"DT-Soft net corrections over majority vote: {net_corrections:+d}")

## 12. Inspect what the tree learned

Feature importance reveals which model-class probabilities drive the decision tree. The rendered tree is useful for interpretability, but a very large tree with no accuracy gain is a warning sign rather than a publication advantage.

In [ ]:
importance = pd.read_csv(run_dir / "decision_tree_feature_importance.csv")
top_importance = importance[importance["importance"] > 0].head(20)
display(top_importance)
if not top_importance.empty:
    plot_data = top_importance.sort_values("importance")
    plot_data.plot.barh(x="feature", y="importance", figsize=(10, 6), legend=False, title="Top DT-Soft feature importances")
    plt.tight_layout()
    plt.show()
display(Image(filename=str(figure_dir / "decision_tree_soft.png"), width=1200))

## 13. Compare final errors

The confusion matrices show whether DT-Soft improves difficult classes or merely moves errors between them. Compare it directly with soft voting and logistic stacking.

In [ ]:
for method, title in [
    ("soft_vote", "Soft Vote"),
    ("logistic_stack", "Logistic Stack"),
    ("dt_soft", "DT-Soft"),
]:
    image_path = figure_dir / f"confusion_{method}.png"
    print(title)
    display(Image(filename=str(image_path), width=650))

## 14. Publication-readiness checks

These are diagnostic gates, not a guarantee of acceptance. They prevent a weak one-seed result from being mistaken for evidence. Failed gates point to the next experiment; they must not be fixed by tuning on the final test set.

In [ ]:
metrics = report["metrics"]
base_accuracies = {name: metrics[name]["accuracy"] for name in ("cnn_1", "cnn_2", "cnn_3")}
strongest_base = max(base_accuracies.values())
dt_soft_accuracy = metrics["dt_soft"]["accuracy"]
soft_vote_accuracy = metrics["soft_vote"]["accuracy"]
logistic_accuracy = metrics["logistic_stack"]["accuracy"]
base_floor = 0.88 if DATASET == "fashion_mnist" else 0.70
gates = [
    ("Every base CNN reaches the diagnostic floor", min(base_accuracies.values()) >= base_floor, f"minimum={min(base_accuracies.values()):.4f}, floor={base_floor:.2f}"),
    ("DT-Soft beats Soft Vote", dt_soft_accuracy > soft_vote_accuracy, f"delta={dt_soft_accuracy - soft_vote_accuracy:+.4f}"),
    ("DT-Soft beats the strongest CNN", dt_soft_accuracy > strongest_base, f"delta={dt_soft_accuracy - strongest_base:+.4f}"),
    ("DT-Soft is at least as strong as Logistic Stack", dt_soft_accuracy >= logistic_accuracy, f"delta={dt_soft_accuracy - logistic_accuracy:+.4f}"),
    ("At least three independent seeds are complete", len(SEEDS) >= 3, f"seeds={len(SEEDS)}"),
]
gate_table = pd.DataFrame(gates, columns=["gate", "passed", "evidence"])
display(gate_table)
if gate_table["passed"].all():
    print("All diagnostic gates passed. Next: repeat both datasets and report uncertainty/significance.")
else:
    print("Not publication-ready yet. Treat failed gates as experiment-design feedback, not as permission to tune on test labels.")

## 15. Save the artifacts

Kaggle preserves files under `/kaggle/working` when you save a notebook version. The zip below is convenient for manual download or attachment to a paper workspace.

In [ ]:
import shutil

archive_base = WORK_ROOT / f"treestack-{DATASET}-results"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print("Download or preserve:", archive_path)

## 16. Publication run settings

Once the one-seed Fashion-MNIST run has converged, use three seeds. A reasonable starting budget is 25 epochs for Fashion-MNIST and 120 for CIFAR-10. Keep test data untouched even if an expected result is not reached; architecture or training changes must be chosen from the base-validation results, followed by a fresh final run.

```python
SEEDS = [17, 42, 73]
# Fashion-MNIST
DATASET, EPOCHS = "fashion_mnist", 25
# CIFAR-10 (run in a separate saved notebook version)
DATASET, EPOCHS = "cifar10", 120
```